# 131 — A2A, descubrimiento e interoperabilidad

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Ver celda: lo evaluable es que la card sea autosuficiente para
decidir la delegación (skill con modos, auth, versión) sin revelar implementación.

**Ejercicio 2.**

```text
submitted → working
working   → input-required | completed | failed | canceled
input-required → working   (el cliente responde)
terminales: completed, failed, canceled
cancelación del cliente: desde submitted, working, input-required
la pregunta de vuelta del remoto: working → input-required
```

**Ejercicio 3.** El filtro es una conjunción de requisitos del contrato; el desempate
por versión es una *política* del cliente (podría ser SLA medido, coste o
reputación — la card sola no informa calidad).

**Ejercicio 4.** `result.workers[security]` se convierte en el artefacto: el
*entregable tipado*. La correspondencia: los `finding` intermedios y la conversación
supervisor↔worker serían `history` (messages); el contrato final
`{agent, score, finding}` es `artifacts`. La distinción del laboratorio entre
`result` (entregable) y `evidence` (hechos del proceso) replica la separación
artifact/history de A2A.


In [ ]:
result = run_lab("multiagent", seed=131)
assert result["kind"] == "multiagent"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1
agent_card = {
    "name": "repo-security-agent",
    "description": "Evalúa la postura de seguridad de repositorios (threat model, secretos, dependencias)",
    "url": "https://agents.example.com/a2a/v1",
    "version": "1.0.0",
    "capabilities": {"streaming": False, "pushNotifications": False},
    "skills": [{
        "id": "repo-security-review",
        "name": "Revisión de seguridad de repositorio",
        "description": "Audita un repositorio y devuelve {agent, score, finding}",
        "inputModes": ["text/plain"],
        "outputModes": ["application/json"],
    }],
    "securitySchemes": {"bearer": {"type": "http", "scheme": "bearer"}},
}

# Ejercicio 3
def elegir_agente(cards, requisitos):
    aptas = []
    for c in cards:
        skills = {s["id"]: s for s in c.get("skills", [])}
        s = skills.get(requisitos["skill_id"])
        if not s or requisitos["output"] not in s.get("outputModes", []):
            continue
        esquemas = {v.get("scheme") for v in c.get("securitySchemes", {}).values()}
        if requisitos["auth"] not in esquemas:
            continue
        aptas.append(c)
    return max(aptas, key=lambda c: tuple(map(int, c["version"].split(".")))) if aptas else None

otra1 = {**agent_card, "name": "sin-json", "version": "3.0.0",
         "skills": [{**agent_card["skills"][0], "outputModes": ["text/plain"]}]}
otra2 = {**agent_card, "name": "sin-auth", "version": "2.0.0", "securitySchemes": {}}
elegida = elegir_agente([otra1, otra2, agent_card],
                        {"skill_id": "repo-security-review",
                         "output": "application/json", "auth": "bearer"})
print("elegida:", elegida["name"], elegida["version"])

# Ejercicio 4
result = run_lab("multiagent", seed=131)
sec = next(w for w in result["result"]["workers"] if w["agent"] == "security")
task_a2a = {
    "id": "T-131", "contextId": "repo-demo",
    "status": {"state": "completed"},
    "history": [{"role": "user", "parts": [{"kind": "text",
                 "text": "evaluar seguridad del repo demo"}]}],
    "artifacts": [{"name": "informe-seguridad",
                    "parts": [{"kind": "data", "data": sec}]}],
}
show(task_a2a)


## Reflexión

1. Los workers del laboratorio son funciones locales del mismo proceso. Si `security` fuera un agente A2A de otra empresa, ¿qué elementos nuevos aparecen en el flujo (descubrimiento, auth, estados, validación del artefacto) y cuál te parece el más frágil?
2. ¿Por qué el estado `input-required` es imprescindible entre organizaciones, y qué patrón de la clase 123 (handoffs) generaliza?
3. La Agent Card declara capacidades pero no calidad. ¿Cómo decidirías entre dos agentes remotos con cards casi idénticas?
